In [0]:
dbutils.widgets.text("catalog","fk_1361")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.read.table(f"{catalog}.bronze.orders_bronze")

In [0]:
df.display()

In [0]:
df_orders = df.withColumns({
    "order_id": F.trim(F.upper("order_id")),
    "customer_id": F.trim(F.upper(F.col("customer_id"))),
    "store_id" : F.trim(F.upper(F.col("store_id"))),
    "order_ts" : F.coalesce(F.try_to_timestamp("order_ts", F.lit("yyyy-MM-dd HH:mm:ss")), F.try_to_timestamp("order_ts", F.lit("yyyy-MM-dd'T'HH:mm:ss"))), 
    # "order_date" : F.to_date(F.col("order_ts"), "yyyy-MM-dd"),
    "order_status" : F.trim(F.upper(F.col("order_status"))),
    "payment_method" : F.trim(F.upper(F.get_json_object("payment", "$.method"))),
    "promised_mins" : F.get_json_object("delivery", "$.promised_mins").cast("int"),
    "actual_mins" : F.get_json_object("delivery", "$.actual_mins").cast("int"),
    "delivery_fee" : F.get_json_object("delivery", "$.delivery_fee").cast("double"),
    "update_at" : F.try_to_timestamp("updated_at", F.lit("yyyy-MM-dd HH:mm:ss"))
    }) \
    .select("order_id","customer_id","store_id","order_ts","order_status","payment_method","promised_mins","actual_mins","delivery_fee","update_at")


In [0]:
df_orders.write.mode("overwrite").saveAsTable(f"{catalog}.silver.orders_silver")

In [0]:
df_expode = df.withColumn("items", F.explode(F.from_json(F.col("items"), "ARRAY<STRUCT<product_id: STRING, qty: INT, unit_price: DOUBLE, discount: DOUBLE>>")))

In [0]:
df_order_items = df_expode.withColumns({
    "order_id" : F.trim(F.upper(F.col("order_id"))),
    "product_id" : F.col("items.product_id"),
    "quantity" : F.col("items.qty").cast("int"),
    "unit_price" : F.col("items.unit_price").cast("double"),
    "discount_amount" : F.coalesce(F.col("items.discount").cast("double"), F.lit(0)),
    "gross_amount" : F.col("unit_price") * F.col("quantity"),
    "net_amount" : F.col("gross_amount") - F.col("discount_amount")
}) \
    .filter((F.col("quantity") > 0) & (F.col("unit_price") > 0)) \
    .dropDuplicates() \
    .dropna(subset = ["customer_id","store_id","order_ts","items","product_id"]) \
    .select("order_id","product_id", "quantity","unit_price","discount_amount","gross_amount","net_amount")

In [0]:
df_order_items.write.mode("overwrite").saveAsTable(f"{catalog}.silver.order_items_silver")